# Woche 11 – Donnerstag: SHAP auf Autoencoder & Vergleich SHAP vs. LIME

## Block 1 (06:00 – 08:45): SHAP auf Autoencoder (Anomalieerkennung)

**Lernziel:** Sie erklären die Anomalieerkennung Ihres Autoencoders mit SHAP.

**Hinweis:** SHAP `DeepExplainer` funktioniert mit PyTorch‑Modellen, ist aber rechenintensiv. Für einfache lineare Autoencoder ist `KernelExplainer` eine Alternative.


In [ ]:
import shap
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Daten vorbereiten (wie in Woche 9)
df = pd.read_json('cleaned_aml_batch.json')
features = df[['amount']].copy()
features['amount_log'] = np.log1p(features['amount'])
scaler = StandardScaler()
X = scaler.fit_transform(features)

# Einfacher linearer Autoencoder
class AE(nn.Module):
    def __init__(self, input_dim=2, bottleneck_dim=1):
        super().__init__()
        self.encoder = nn.Linear(input_dim, bottleneck_dim)
        self.decoder = nn.Linear(bottleneck_dim, input_dim)
    def forward(self, x):
        return self.decoder(self.encoder(x))

model = AE()
model.load_state_dict(torch.load('best_ae_model.pth'))  # aus Woche 9
model.eval()

# SHAP KernelExplainer (modell‑agnostisch)
background = X[:100]  # Hintergrunddatensatz für KernelExplainer
explainer = shap.KernelExplainer(model, background)
shap_values = explainer.shap_values(X[:10])  # nur 10 Instanzen (langsam)

shap.summary_plot(shap_values, X[:10], feature_names=['amount', 'amount_log'])


> **Verbesserungshinweis:** `KernelExplainer` ist rechenintensiv – für viele Instanzen ist `DeepExplainer` für PyTorch schneller (aber erfordert mehr Einrichtung).

---

## Block 2 (09:30 – 11:40): Vergleich SHAP vs. LIME

**Aufgabe:** Diskutieren Sie die Vor‑ und Nachteile beider Methoden in STEM. Erstellen Sie eine Tabelle:

| Kriterium | SHAP | LIME |
|-----------|------|------|
| Globale vs. lokale Erklärung | Beides (global aggregiert) | Hauptsächlich lokal |
| Theoretische Fundierung | Shapley‑Werte (spieltheoretisch) | Lokale lineare Approximation |
| Rechenaufwand | Hoch (besonders KernelExplainer) | Mittel |
| Konsistenz | Ja (konsistent) | Nein (instabil) |
| Modell‑Agnostisch | Ja (mit KernelExplainer) | Ja |

> **Verbesserungshinweis:** SHAP ist theoretisch fundierter, aber rechenintensiver. LIME ist schneller, aber die Erklärungen können je nach Stichprobe variieren.

---

## Block 3 (12:40 – 14:10): GOV – Transparenzbericht für die Aufsicht

**Aufgabe:** Verfassen Sie einen einseitigen Bericht an die BaFin (Bundesanstalt für Finanzdienstleistungsaufsicht) mit dem Titel: „Nachweis der Transparenz unseres AML‑Autoencoders gemäß Art. 50“. Gehen Sie darauf ein, wie SHAP/LIME zur Erklärbarkeit beitragen. Speichern Sie als `63_transparenzbericht_bafin.md` in Track_C.

---

## Block 4 (14:40 – 16:50): SPR – Abschlussübung Passiv und Passiv‑Ersatzformen

Schreiben Sie einen Aufsatz (1 Seite) über die Implementierung von XAI in Ihrem AML‑System. Verwenden Sie ausschließlich das Vorgangspassiv und Passiv‑Ersatzformen (kein Aktiv). Speichern Sie als `64_passiv_xai_abschluss.md` in Track_C.